# Chunking Strategy Test

Tests **5 chunking strategies** against the FastAPI corpus.  
No LLM, no GPU, no API key needed — runs in ~5 minutes on CPU.

## How to add your own strategy
1. Write a function `chunk_<name>(corpus) -> list[dict]` in **Section 2**
2. Add it to the `STRATEGIES` dict at the bottom of Section 2
3. Re-run from Section 3 onwards — all charts update automatically

## Strategies included
| # | Name | Idea |
|---|---|---|
| 1 | `ast_current` | One chunk per function (your baseline) |
| 2 | `ast_class_aware` | Prepend parent class name to method chunks |
| 3 | `ast_filtered` | Drop trivially small chunks (<10 tokens) |
| 4 | `sliding_window` | 200-token windows with 50-token overlap |
| 5 | `summary_enhanced` | Prepend English summary to chunk text |

## Metrics (all LLM-free)
| Metric | What it measures |
|---|---|
| `hit_rate@k` | Did any golden chunk appear in top-k? |
| `mrr@k` | How high up was the first correct result? |
| `precision@k` | Of what was retrieved, how much was relevant? |
| `recall@k` | Of all golden chunks, how many were found? |

## 0 · Install

In [ ]:
%pip install -q sentence-transformers rank_bm25 pandas matplotlib seaborn scikit-learn tqdm

## 1 · Load data

In [ ]:
import json, re, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

DATA_DIR = Path(".")

# ── Load corpus and golden set ────────────────────────────────────────────────
with open(DATA_DIR / "local_corpus.json") as f:
    corpus = json.load(f)

with open(DATA_DIR / "ragas_dataset.jsonl") as f:
    ragas_records = [json.loads(l) for l in f]

with open(DATA_DIR / "fastapi_golden_set_splits.json") as f:
    splits = json.load(f)

# Use the test split only — no data leakage
test_questions = {x["instruction"] for x in splits["test"]}
test_records   = [r for r in ragas_records if r["question"] in test_questions]

# Quick eval: 20 records. Set QUICK=False for the full 85-record test split.
QUICK = True
QUICK_N = 20
random.seed(42)
eval_records = random.sample(test_records, QUICK_N) if QUICK else test_records

# Embedder — same one used by rag_pipeline_fixed.py
print("Loading embedder...")
EMBEDDER = SentenceTransformer(
    "flax-sentence-embeddings/st-codesearch-distilroberta-base"
)

print(f"Corpus       : {len(corpus)} chunks")
print(f"Test records : {len(eval_records)}")
print("✅ Ready")

## 2 · Chunking strategies

Each strategy is a function that takes the corpus and returns a list of chunks.
Each chunk is a dict with at minimum: `{"id": str, "content": str}`.

**To add your own:** copy Strategy 5 as a template, change the logic, add to `STRATEGIES`.

In [ ]:
# ── Helpers used by multiple strategies ──────────────────────────────────────

def has_chinese(text: str) -> bool:
    return any("\u4e00" <= c <= "\u9fff" for c in text)

def source_file(doc_id: str) -> str:
    """'FastAPI-Core_docs_src_security_tutorial001_py310.py_f0' -> same without _f0"""
    return re.sub(r"_f\d+$", "", doc_id)

def extract_fn_name(code: str) -> str | None:
    m = re.search(r"(?:async )?def (\w+)", code)
    return m.group(1) if m else None

def is_method(code: str) -> bool:
    """True if the function takes 'self' as first param."""
    return bool(re.search(r"def \w+\(\s*self", code))

print("Helpers defined.")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Strategy 1: ast_current  (YOUR BASELINE — do not modify)
# One chunk per AST function.  This is exactly what rag_pipeline_fixed.py
# already uses.  All other strategies are compared against this.
# ════════════════════════════════════════════════════════════════════════
def chunk_ast_current(corpus: list[dict]) -> list[dict]:
    return [
        {"id": x["id"], "content": x["content"]}
        for x in corpus
    ]


# ════════════════════════════════════════════════════════════════════════
# Strategy 2: ast_class_aware
# Problem being solved:
#   __init__(self, tokenUrl=...) tells you NOTHING without knowing the class.
#   We lost that context when we stripped methods from their class body.
# Fix:
#   Detect methods (have 'self'), then extract the source file name as a
#   proxy for the class name and prepend it.
# ════════════════════════════════════════════════════════════════════════
def chunk_ast_class_aware(corpus: list[dict]) -> list[dict]:
    chunks = []
    for x in corpus:
        code = x["content"]
        if is_method(code):
            # Derive a class hint from the source file path
            # e.g. 'fastapi_security_oauth2.py' -> 'security_oauth2'
            src = source_file(x["id"])
            # Strip prefix and .py
            hint = re.sub(r"^FastAPI-Core_", "", src)
            hint = re.sub(r"\.py$", "", hint).replace("_", ".")
            prefix = f"# class context: {hint}\n"
            content = prefix + code
        else:
            content = code
        chunks.append({"id": x["id"] + "_cls", "content": content})
    return chunks


# ════════════════════════════════════════════════════════════════════════
# Strategy 3: ast_filtered
# Problem being solved:
#   20 chunks are trivially small (<5 words): 'def callback(): pass'
#   These add noise — they score moderately for many queries and
#   dilute the reranker's candidate pool.
# Fix:
#   Drop chunks below MIN_TOKENS words.  Also drop chunks that are
#   pure pass-through stubs (contain only 'pass' or 'return None').
# ════════════════════════════════════════════════════════════════════════
MIN_TOKENS = 10

def chunk_ast_filtered(corpus: list[dict]) -> list[dict]:
    chunks = []
    for x in corpus:
        code = x["content"].strip()
        n_tokens = len(code.split())
        # Skip tiny stubs
        if n_tokens < MIN_TOKENS:
            continue
        # Skip pass-only bodies
        body_lines = [l.strip() for l in code.splitlines()[1:] if l.strip()]
        if all(l in ("pass", "...", "return None", "return") for l in body_lines):
            continue
        chunks.append({"id": x["id"] + "_flt", "content": code})
    return chunks


# ════════════════════════════════════════════════════════════════════════
# Strategy 4: sliding_window
# Problem being solved:
#   The 959-line FastAPI.__init__ chunk is a single vector that has to
#   represent everything — it scores mediocrely for every specific query.
#   Very small chunks (median 5 lines) lose surrounding context.
# Fix:
#   Reassemble each source file in order, then apply a sliding window
#   of WINDOW words with OVERLAP words of overlap between adjacent chunks.
#   This gives every token multiple chances to appear in a chunk.
# ════════════════════════════════════════════════════════════════════════
WINDOW  = 200   # words per window
OVERLAP = 50    # words of overlap between adjacent windows

def chunk_sliding_window(corpus: list[dict]) -> list[dict]:
    # Group by source file, sort by chunk index to preserve code order
    by_file: dict[str, list[dict]] = defaultdict(list)
    for x in corpus:
        by_file[source_file(x["id"])].append(x)
    for chunks in by_file.values():
        chunks.sort(key=lambda x: int(re.search(r"_f(\d+)$", x["id"]).group(1)))

    result = []
    for file_id, chunks in by_file.items():
        # Concatenate all functions in this file
        full_text = "\n\n".join(c["content"] for c in chunks)
        words     = full_text.split()

        i   = 0
        idx = 0
        while i < len(words):
            window_words = words[i : i + WINDOW]
            result.append({
                "id":      f"{file_id}_win{idx}",
                "content": " ".join(window_words),
            })
            i   += WINDOW - OVERLAP
            idx += 1

    return result


# ════════════════════════════════════════════════════════════════════════
# Strategy 5: summary_enhanced
# Problem being solved:
#   Concept queries like 'share dependencies across routes' have zero
#   lexical overlap with 'common_parameters'.
#   English summaries (23% of corpus) describe WHAT the function does,
#   which is exactly what concept queries ask.
# Fix:
#   Prepend the English summary to the code so both the concept and the
#   implementation are in the same embedding space.
#   Chinese summaries are skipped (they hurt the embedding).
# ════════════════════════════════════════════════════════════════════════
def chunk_summary_enhanced(corpus: list[dict]) -> list[dict]:
    chunks = []
    for x in corpus:
        summary = x["metadata"].get("summary", "")
        code    = x["content"]
        if summary and not has_chinese(summary):
            # Strip the [filepath] prefix from the summary
            clean = re.sub(r"^\[[^\]]+\]\s*", "", summary).strip()
            # Only prepend if it adds real information (not just the function signature)
            fn_name = extract_fn_name(code) or ""
            if clean and fn_name not in clean[:40]:
                content = f"# {clean}\n{code}"
            else:
                content = code
        else:
            content = code
        chunks.append({"id": x["id"] + "_sum", "content": content})
    return chunks


# ════════════════════════════════════════════════════════════════════════
# ➕ ADD YOUR STRATEGY HERE
# Copy the template below, fill in the logic, add to STRATEGIES dict.
# ════════════════════════════════════════════════════════════════════════
# def chunk_my_strategy(corpus: list[dict]) -> list[dict]:
#     """
#     Describe what problem this solves and how.
#     """
#     chunks = []
#     for x in corpus:
#         # ... your chunking logic ...
#         chunks.append({"id": x["id"] + "_my", "content": modified_content})
#     return chunks


# ── Register all strategies ───────────────────────────────────────────────────
# Key   = display name in charts
# Value = the chunking function
STRATEGIES = {
    "1_ast_current":     chunk_ast_current,
    "2_ast_class_aware": chunk_ast_class_aware,
    "3_ast_filtered":    chunk_ast_filtered,
    "4_sliding_window":  chunk_sliding_window,
    "5_summary_enhanced":chunk_summary_enhanced,
    # "6_my_strategy":  chunk_my_strategy,   # ← add yours here
}

# Build all chunk sets
print("Building chunk sets...")
chunk_sets = {}
for name, fn in STRATEGIES.items():
    chunks = fn(corpus)
    chunk_sets[name] = chunks
    print(f"  {name:25s}: {len(chunks):5d} chunks")

## 3 · Build retrievers

In [ ]:
# ── Dense retriever using the same embedder as rag_pipeline_fixed.py ─────────
class DenseRetriever:
    def __init__(self, chunks: list[dict]):
        self.chunks = chunks
        texts = [c["content"] for c in chunks]
        self.embeddings = EMBEDDER.encode(
            texts, batch_size=64, show_progress_bar=False
        )

    def retrieve(self, query: str, top_k: int) -> list[str]:
        q_emb = EMBEDDER.encode([query])
        sims  = cosine_similarity(q_emb, self.embeddings)[0]
        top_i = np.argsort(sims)[::-1][:top_k]
        return [self.chunks[i]["content"] for i in top_i]


# Build one retriever per strategy
print("Building retrievers (this takes ~1 min)...")
retrievers = {}
for name, chunks in chunk_sets.items():
    print(f"  Embedding {name} ({len(chunks)} chunks)...", end="", flush=True)
    retrievers[name] = DenseRetriever(chunks)
    print(" done")

print("\n✅ All retrievers ready")

## 4 · Evaluation metrics

In [ ]:
def fingerprint(text: str) -> str:
    """Fast content key — strips whitespace, takes first 60 chars."""
    return re.sub(r"\s+", "", text)[:60]

def golden_fn_names(contexts: list[str]) -> set[str]:
    """Extract function names from golden contexts for semantic matching."""
    fns = set()
    for ctx in contexts:
        fns |= set(re.findall(r"(?:async )?def (\w+)", ctx))
    return fns - {"__init__", "__call__", "main", "app"}

def is_hit(retrieved: list[str], golden: list[str]) -> bool:
    """
    Semantic hit: a retrieved chunk is a hit if:
      - its fingerprint matches a golden chunk exactly, OR
      - it defines the same function name as a golden chunk
        (handles tutorial variant files correctly)
    """
    golden_fps = {fingerprint(g) for g in golden}
    golden_fns = golden_fn_names(golden)
    for r in retrieved:
        if fingerprint(r) in golden_fps:
            return True
        if golden_fns:
            ret_fns = set(re.findall(r"(?:async )?def (\w+)", r))
            if golden_fns & ret_fns:
                return True
    return False

def hit_rate(retrieved, golden):  return float(is_hit(retrieved, golden))

def mrr(retrieved, golden):
    golden_fps = {fingerprint(g) for g in golden}
    golden_fns = golden_fn_names(golden)
    for rank, r in enumerate(retrieved, 1):
        if fingerprint(r) in golden_fps:
            return 1.0 / rank
        if golden_fns:
            ret_fns = set(re.findall(r"(?:async )?def (\w+)", r))
            if golden_fns & ret_fns:
                return 1.0 / rank
    return 0.0

def precision_at_k(retrieved, golden):
    if not retrieved: return 0.0
    golden_fps = {fingerprint(g) for g in golden}
    golden_fns = golden_fn_names(golden)
    hits = sum(
        1 for r in retrieved
        if fingerprint(r) in golden_fps
        or (golden_fns & set(re.findall(r"(?:async )?def (\w+)", r)))
    )
    return hits / len(retrieved)

def recall_at_k(retrieved, golden):
    if not golden: return 0.0
    golden_fns = golden_fn_names(golden)
    if not golden_fns:
        golden_fps = {fingerprint(g) for g in golden}
        hits = sum(1 for r in retrieved if fingerprint(r) in golden_fps)
        return hits / len(golden_fps)
    ret_fns = set(fn for r in retrieved for fn in re.findall(r"(?:async )?def (\w+)", r))
    return len(golden_fns & ret_fns) / len(golden_fns)

print("✅ Metrics defined")

## 5 · Run evaluation

In [ ]:
TOP_K_VALUES = [1, 3, 5, 10]
rows = []

for strategy_name, retriever in retrievers.items():
    for r in tqdm(eval_records, desc=strategy_name[:25], leave=False):
        question = r["question"]
        golden   = r["contexts"]
        topic    = r["_meta"]["topic"]

        for k in TOP_K_VALUES:
            retrieved = retriever.retrieve(question, top_k=k)
            rows.append({
                "strategy":  strategy_name,
                "k":         k,
                "topic":     topic,
                "hit_rate":  hit_rate(retrieved, golden),
                "mrr":       mrr(retrieved, golden),
                "precision": precision_at_k(retrieved, golden),
                "recall":    recall_at_k(retrieved, golden),
            })

df = pd.DataFrame(rows)
print(f"\n{len(df)} evaluation rows computed.")
print(df.groupby(["strategy", "k"])[["hit_rate","mrr","precision","recall"]].mean().round(3))

## 6 · Charts

In [ ]:
# ── Summary table at k=5 ─────────────────────────────────────────────────────
summary = (
    df[df["k"] == 5]
    .groupby("strategy")[["hit_rate", "mrr", "precision", "recall"]]
    .mean().round(3)
    .sort_values("recall", ascending=False)
)

palette = {
    "1_ast_current":      "#B4B2A9",
    "2_ast_class_aware":  "#7F77DD",
    "3_ast_filtered":     "#EF9F27",
    "4_sliding_window":   "#1D9E75",
    "5_summary_enhanced": "#D85A30",
}

print("Results at k=5")
print("─" * 60)
print(summary.to_string())
print()
best = summary["recall"].idxmax()
print(f"Best strategy by recall: {best}  ({summary.loc[best, 'recall']:.1%})")

In [ ]:
# ── Chart 1: All 4 metrics at k=5 ────────────────────────────────────────────
df5      = df[df["k"] == 5]
metrics  = ["hit_rate", "mrr", "precision", "recall"]
titles   = ["Hit rate@5", "MRR@5", "Precision@5", "Recall@5"]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, metric, title in zip(axes, metrics, titles):
    means = df5.groupby("strategy")[metric].mean().reindex(list(palette))
    bars  = ax.bar(
        range(len(means)), means.values,
        color=[palette.get(s, "#888") for s in means.index],
        width=0.55, edgecolor="white"
    )
    ax.set_title(title, fontsize=11)
    ax.set_ylim(0, 1)
    ax.set_xticks(range(len(means)))
    ax.set_xticklabels(
        [s.split("_", 1)[1] for s in means.index],
        rotation=30, ha="right", fontsize=8
    )
    ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
    for bar, val in zip(bars, means.values):
        if not np.isnan(val):
            ax.text(bar.get_x() + bar.get_width()/2, val + 0.02,
                    f"{val:.2f}", ha="center", fontsize=8)

plt.suptitle("Chunking strategy comparison at k=5", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(DATA_DIR / "chunking_metrics_k5.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: chunking_metrics_k5.png")

In [ ]:
# ── Chart 2: Recall@k curve — find the optimal top_k ─────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
for name, color in palette.items():
    recall_by_k = [
        df[(df["strategy"] == name) & (df["k"] == k)]["recall"].mean()
        for k in TOP_K_VALUES
    ]
    label = name.split("_", 1)[1]
    ax.plot(TOP_K_VALUES, recall_by_k, marker="o", label=label,
            color=color, linewidth=2)

ax.set_xlabel("k (chunks retrieved)")
ax.set_ylabel("Recall@k")
ax.set_title("Recall@k — where does each strategy plateau?")
ax.set_ylim(0, 1)
ax.set_xticks(TOP_K_VALUES)
ax.axhline(0.7, color="gray", linestyle="--", linewidth=0.8, label="0.7 target")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(DATA_DIR / "chunking_recall_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: chunking_recall_curve.png")
print("Tip: the k where the curve flattens is your optimal top_k for that strategy")

In [ ]:
# ── Chart 3: Per-topic recall heatmap ────────────────────────────────────────
pivot = (
    df[df["k"] == 5]
    .groupby(["topic", "strategy"])["recall"]
    .mean().round(3)
    .unstack("strategy")
    .dropna(how="all")
    .reindex(columns=list(palette))
)
pivot.columns = [c.split("_", 1)[1] for c in pivot.columns]

fig, ax = plt.subplots(figsize=(12, max(4, len(pivot) * 0.4)))
sns.heatmap(
    pivot.fillna(0), annot=True, fmt=".2f",
    cmap="YlGn", vmin=0, vmax=1,
    linewidths=0.3, ax=ax
)
ax.set_title("Recall@5 per FastAPI topic — spot where each strategy struggles")
plt.tight_layout()
plt.savefig(DATA_DIR / "chunking_topic_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: chunking_topic_heatmap.png")

In [ ]:
# ── Chart 4: Precision vs Recall trade-off ────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
for name, color in palette.items():
    p_vals = [df[(df["strategy"]==name)&(df["k"]==k)]["precision"].mean() for k in TOP_K_VALUES]
    r_vals = [df[(df["strategy"]==name)&(df["k"]==k)]["recall"].mean()    for k in TOP_K_VALUES]
    label  = name.split("_", 1)[1]
    ax.plot(r_vals, p_vals, marker="o", label=label, color=color, linewidth=2)
    for k, r, p in zip(TOP_K_VALUES, r_vals, p_vals):
        ax.annotate(f"k={k}", (r, p), xytext=(4, 3),
                    textcoords="offset points", fontsize=7, color=color)

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision–Recall trade-off by k")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(DATA_DIR / "chunking_pr_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: chunking_pr_curve.png")
print("Best chunking = upper-right corner at the lowest k value")

## 7 · Chunk size analysis

In [ ]:
# Show token count distribution per strategy — helps diagnose size issues
fig, axes = plt.subplots(1, len(STRATEGIES), figsize=(4*len(STRATEGIES), 3), sharey=True)
colors = list(palette.values())

for ax, (name, chunks), color in zip(axes, chunk_sets.items(), colors):
    sizes = [len(c["content"].split()) for c in chunks]
    ax.hist(sizes, bins=30, color=color, edgecolor="white")
    ax.set_title(f"{name.split('_',1)[1]}\nn={len(chunks)}, med={int(np.median(sizes))}",
                 fontsize=9)
    ax.set_xlabel("tokens", fontsize=8)
    ax.axvline(200, color="red", linestyle="--", linewidth=0.8)

axes[0].set_ylabel("count")
plt.suptitle("Chunk size distribution (red = 200 tokens)", y=1.02)
plt.tight_layout()
plt.savefig(DATA_DIR / "chunking_size_dist.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: chunking_size_dist.png")
print("Chunks below ~20 tokens are usually too small; above ~400 are too noisy for embeddings")

## 8 · Save results

In [ ]:
df.to_csv(DATA_DIR / "chunking_results.csv", index=False)
summary.to_csv(DATA_DIR / "chunking_summary.csv")

print("Saved: chunking_results.csv")
print("Saved: chunking_summary.csv")
print()
print("Final ranking by recall@5:")
print("─" * 50)
for i, (name, row) in enumerate(summary.iterrows(), 1):
    delta = row["recall"] - summary.loc["1_ast_current", "recall"]
    arrow = "▲" if delta > 0.02 else ("▼" if delta < -0.02 else "≈")
    print(f"  {i}. {name:25s}  recall={row['recall']:.3f}  {arrow} {delta:+.3f} vs baseline")